# TabPFN → drzewo decyzyjne (rozwiązanie 2)

Colab: **Runtime → Change runtime type → T4 GPU**, potem Run all.

Pipeline destylacji:

1. **TabPFN** trenuje się na etykietach: `val.csv`, albo `val_full.csv` gdy `FINAL_EVALUATION_MODE=True`. `test.csv` nigdy nie wchodzi do fitu.
2. Nauczyciel **dopisuje etykiety** do nieoznaczonego `train.csv` (pseudo-labelki).
3. **Student (drzewo)** uczy się ze znacznie większego zbioru: `val` ∪ `train`.

W notebooku są **dwa** drzewa na tym samym nauczycielu, żeby porównać z wariantem „tylko val”. Aplikacja (`app_tabpfn.py`) dostaje drzewo z powiększonego zbioru — CPU, ścieżka if/then.

**Weryfikacja (`FINAL_EVALUATION_MODE = False`):** uczciwy Raw_Score na `final_valid.csv` z wstrzykniętymi lukami (~5%, jak `test.csv`).

**Submit (`FINAL_EVALUATION_MODE = True`):** trening na `val_full.csv` (= `val.csv` ∪ `final_valid.csv`). Holdoutu już nie ma — nie liczymy score, tylko predykcje na `test.csv`.

In [6]:
import sys
from pathlib import Path


IN_COLAB = "google.colab" in sys.modules
IN_COLAB = False
if IN_COLAB:
    %pip install -q tabpfn scikit-learn pandas numpy torch joblib plotly
    from google.colab import files
    print("Wgraj val.csv, val_full.csv, final_valid.csv, train.csv, test.csv oraz tabpfn_diagnose.py")
    files.upload()

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

cuda: False CPU


In [7]:
from tabpfn_diagnose import (
    TabPFNTreeDiagnoser,
    hackathon_score,
    pick_device,
    print_eval,
    punch_spectrum_gaps,
    read_labeled_csv,
)
import pandas as pd

# True  → fit na val_full.csv (val ∪ final_valid), bez scorowania holdoutu, submit test.csv
# False → fit na val.csv, uczciwy Raw_Score na final_valid.csv (+ porównanie z zwykłym drzewem)
FINAL_EVALUATION_MODE = False

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

if FINAL_EVALUATION_MODE:
    val = read_labeled_csv("val_full.csv")
    holdout = None
    print("FINAL_EVALUATION_MODE=True  trening = val_full.csv  (holdout wchłonięty, nie ewaluuję)")
    print(
        "val_full", val.shape,
        "silniki:", val["engine_id"].nunique(),
        "train", train.shape, "test", test.shape,
        "device=", pick_device(),
    )
    print("symulacja luk warsztatowych na val_full (train/test już je mają):")
    val = punch_spectrum_gaps(val, verbose=True)
else:
    val = read_labeled_csv("val.csv")
    holdout = read_labeled_csv("final_valid.csv")
    overlap = set(val["engine_id"]) & set(holdout["engine_id"])
    assert not overlap, f"wyciek silników val ∩ final_valid: {sorted(overlap)}"
    print("FINAL_EVALUATION_MODE=False  trening = val.csv  holdout = final_valid.csv")
    print(
        "val", val.shape, "final_valid", holdout.shape,
        "train", train.shape, "test", test.shape,
        "device=", pick_device(),
    )
    print("silniki holdout:", sorted(holdout["engine_id"].unique()))
    print("symulacja luk warsztatowych na val / final_valid (train/test już je mają):")
    val = punch_spectrum_gaps(val, verbose=True)
    holdout = punch_spectrum_gaps(holdout, verbose=True)

punch_spectrum_gaps(train, verbose=True)
punch_spectrum_gaps(test, verbose=True)

FINAL_EVALUATION_MODE=True  trening = val_full.csv  (holdout wchłonięty, nie ewaluuję)
val_full (476, 26) silniki: 40 train (2400, 24) test (600, 24) device= cpu
symulacja luk warsztatowych na val_full (train/test już je mają):
  luki pomiarowe: 483/9996 (4.83%)  wiersze z luką: 301/476
  luki pomiarowe: już 2525/50400 (5.01%) — nie dubluję
  luki pomiarowe: już 613/12600 (4.87%) — nie dubluję


,engine_id,cylinder,n_cylinders,mV_0,mV_1,mV_2,mV_3,mV_4,mV_5,mV_6,...,mV_11,mV_12,mV_13,mV_14,mV_15,mV_16,mV_17,mV_18,mV_19,mV_20
0,test_0000,1,12,31.388,27.459,21.718,23.700,32.478,40.316,46.006,...,40.234,33.746,28.787,24.923,22.882,24.746,28.283,31.456,30.684,30.510
1,test_0000,2,12,37.032,31.706,26.391,31.481,40.384,46.839,51.663,...,43.938,36.174,30.475,27.255,26.465,28.765,32.468,NaN,33.366,33.317
2,test_0000,3,12,33.684,28.276,22.274,26.277,35.830,43.646,48.947,...,44.134,37.415,31.745,26.991,24.424,25.548,28.397,32.143,31.559,30.770
3,test_0000,4,12,35.548,33.088,27.087,26.400,34.228,42.246,48.196,...,44.684,37.806,32.140,28.226,26.473,28.195,31.098,33.318,31.506,30.870
4,test_0000,5,12,37.273,33.748,27.608,29.001,37.703,NaN,51.824,...,46.922,39.432,34.006,NaN,28.757,30.621,33.952,36.031,34.425,33.733
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,test_0049,12,16,40.200,36.570,28.102,NaN,42.690,50.595,56.740,...,58.332,54.844,50.219,44.540,37.945,31.138,18.020,18.453,23.179,22.482
596,test_0049,13,16,38.821,35.423,25.482,32.786,41.687,50.599,57.423,...,61.505,57.185,52.066,46.035,38.108,29.656,15.039,17.695,23.061,22.449
597,test_0049,14,16,40.008,33.819,24.721,34.377,43.791,52.340,57.596,...,58.292,52.793,48.520,41.341,34.519,24.348,NaN,23.327,23.092,22.509
598,test_0049,15,16,40.048,35.918,25.909,NaN,42.109,50.237,56.069,...,57.863,52.997,48.721,41.388,34.213,24.071,12.233,22.304,22.772,24.044


## Nauczyciel: TabPFN na `val.csv`

TabPFN widzi zbiór treningowy z wstrzykniętymi lukami jak w `test.csv`. Przy `FINAL_EVALUATION_MODE=False` destylujemy najpierw drzewo tylko z `val` i zapisujemy predykcje holdoutu do porównania. Przy `True` holdoutu nie ma — od razu nauczyciel na `val_full`. Na T4 możesz zostawić `do_cv=True` (GroupKFold po silniku; w trybie finalnym CV jest wyłączone). Na CPU: `do_cv=False`.

In [8]:
device = pick_device()
n_estimators = 8 if device == "cuda" else 4
do_cv = (not FINAL_EVALUATION_MODE) and device == "cuda"  # T4 + tryb walidacji; final/CPU: pomiń
CONF_MIN = 0.70  # pewność pseudo-etykiet; 0.0 = weź cały train.csv

model = TabPFNTreeDiagnoser().fit(
    val,
    train=None,  # student A: tylko etykietowany zbiór
    device=device,
    n_estimators=n_estimators,
    do_cv=do_cv,
)
if FINAL_EVALUATION_MODE:
    sub_ho_tree_val_only = None
    print("FINAL_EVALUATION_MODE: nie przewiduję final_valid (jest w treningu)")
else:
    sub_ho_tree_val_only = model.predict(holdout)
print("student A (tylko labeled)  n_student=", model.meta.get("n_student"))
print(model.meta)

TabPFN teacher  device=cpu  n_estimators=4
TabPFN in-sample on val.csv (sanity, not the holdout score):
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000       407
 zakoksowany      1.000     1.000     1.000        18
      lejacy      1.000     1.000     1.000        14
       pompa      1.000     1.000     1.000         9
      iglica      1.000     1.000     1.000        16
     unknown      1.000     1.000     1.000        12

    accuracy                          1.000       476
   macro avg      1.000     1.000     1.000       476
weighted avg      1.000     1.000     1.000       476

Student tree distilled on 476 rows (val=476, pseudo=0)

Distilled tree on val.csv (resubstitution, not holdout):
              precision    recall  f1-score   support

          ok      0.998     0.998     0.998       407
 zakoksowany      1.000     0.944     0.971        18
      lejacy      1.000     1.000     1.000        14
       pompa      1.000

## Student z powiększonym zbiorem: `val` ∪ `train`

TabPFN (już wytrenowany) etykietuje `train.csv`. Drzewo destylujemy ze **znacznie większego** zbioru: prawdziwe labelki z labeled + pseudo-labelki z train. Nauczyciel się nie trenuje drugi raz.

Przy `FINAL_EVALUATION_MODE=False` porównanie na `final_valid.csv` z lukami:
`zwykłe drzewo ← val` / TabPFN / drzewo destylowane←val / drzewo destylowane←val+pseudo-train.
Zwykłe drzewo = te same cechy akustyczne, prawdziwe etykiety, **bez** TabPFN i bez pseudo-`train`.
Przy `True` skipujemy score — holdout jest w fitu. Do aplikacji zapisujemy wariant z train.

In [9]:
plain_tree = model.fit_plain_tree()  # te same cechy, prawdziwe etykiety val, bez TabPFN
n_fit_val = int(model.meta["n_student"])
model.distill(train, conf_min=CONF_MIN)  # student B: labeled + pseudo-train
n_fit_big = int(model.meta["n_student"])
model.save()

print(
    f"pseudo-labelki: {model.meta.get('n_pseudo')}/{len(train)} "
    f"(próg P ≥ {CONF_MIN:.2f})  →  student {n_fit_val} → {n_fit_big} wierszy"
)
print("rozkład pseudo-etykiet:", model.meta.get("train_pseudo", {}).get("label_counts"))

if FINAL_EVALUATION_MODE:
    print("FINAL_EVALUATION_MODE: nie liczę Raw_Score na final_valid (brak holdoutu)")
else:
    y_ho = holdout["label"].to_numpy()
    s_ho = holdout["severity"].to_numpy()
    sub_ho_plain = model.predict_with(holdout, plain_tree)
    sub_ho_tabpfn = model.predict_teacher(holdout, model.teacher_)
    sub_ho_tree = model.predict(holdout)
    rows = []
    for name, sub, n_fit in [
        ("zwykłe drzewo ← val (bez TabPFN)", sub_ho_plain, len(val)),
        ("TabPFN teacher", sub_ho_tabpfn, len(val)),
        ("drzewo destylowane ← val", sub_ho_tree_val_only, n_fit_val),
        ("drzewo destylowane ← val + pseudo-train", sub_ho_tree, n_fit_big),
    ]:
        raw, macro, sev = hackathon_score(
            y_ho, sub["label"].to_numpy(), s_ho, sub["severity"].to_numpy()
        )
        agree = float((sub["label"].to_numpy() == sub_ho_tabpfn["label"].to_numpy()).mean())
        rows.append(
            {
                "model": name,
                "n_fit": n_fit,
                "Raw_Score": raw,
                "macro-F1": macro,
                "severity_acc": sev,
                "zgoda vs TabPFN": agree,
            }
        )
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    agree_plain = float((sub_ho_plain["label"] == sub_ho_tree_val_only["label"]).mean())
    print(f"zgoda zwykłe drzewo vs destylowane←val: {agree_plain:.3f}  (powinny być prawie identyczne)")

Pseudo-labels from train.csv: 2395/2400 with max P ≥ 0.70
ok             2050
unknown          78
zakoksowany      73
iglica           68
pompa            64
lejacy           62
Student tree distilled on 2871 rows (val=476, pseudo=2395)

Distilled tree on val.csv (resubstitution, not holdout):
              precision    recall  f1-score   support

          ok      1.000     0.998     0.999       407
 zakoksowany      1.000     0.944     0.971        18
      lejacy      1.000     1.000     1.000        14
       pompa      0.900     1.000     0.947         9
      iglica      1.000     1.000     1.000        16
     unknown      0.923     1.000     0.960        12

    accuracy                          0.996       476
   macro avg      0.971     0.990     0.980       476
weighted avg      0.996     0.996     0.996       476

tree vs val labels  macro-F1=0.9796  severity_acc=0.9298  Raw_Score=0.9672  fidelity vs TabPFN=0.996
Wrote /home/janek/Desktop/hackathon-engin/artifacts/diagnoser

## Holdout: `final_valid.csv` — raporty klas

Silniki spoza `val.csv`. Pełny classification_report: zwykłe drzewo, TabPFN, destylacja z val, destylacja z val+train. Przy `FINAL_EVALUATION_MODE=True` ta komórka nic nie liczy.

In [10]:
if FINAL_EVALUATION_MODE:
    print("FINAL_EVALUATION_MODE: brak holdoutu — pomijam raporty na final_valid")
else:
    print_eval("final_valid — zwykłe drzewo ← val", y_ho, sub_ho_plain["label"].to_numpy(), s_ho, sub_ho_plain["severity"].to_numpy())
    print_eval("final_valid — TabPFN teacher", y_ho, sub_ho_tabpfn["label"].to_numpy(), s_ho, sub_ho_tabpfn["severity"].to_numpy())
    print_eval("final_valid — drzewo destylowane ← val + pseudo-train", y_ho, sub_ho_tree["label"].to_numpy(), s_ho, sub_ho_tree["severity"].to_numpy())
    print_eval("final_valid — drzewo destylowane ← val", y_ho, sub_ho_tree_val_only["label"].to_numpy(), s_ho, sub_ho_tree_val_only["severity"].to_numpy())
    print(
        f"zgoda destylowane(val+train) vs TabPFN: {(sub_ho_tree['label'] == sub_ho_tabpfn['label']).mean():.3f}  |  "
        f"zgoda destylowane(val) vs TabPFN: {(sub_ho_tree_val_only['label'] == sub_ho_tabpfn['label']).mean():.3f}  |  "
        f"zgoda zwykłe drzewo vs TabPFN: {(sub_ho_plain['label'] == sub_ho_tabpfn['label']).mean():.3f}"
    )

FINAL_EVALUATION_MODE: brak holdoutu — pomijam raporty na final_valid


## Drzewo, które zobaczy mechanik

In [11]:
print(model.rules_text())

|--- residual 9 kHz vs. baseline silnika <= -8.43
|   |--- podobieństwo do wzorca: zakoksowany <= 0.81
|   |   |--- podobieństwo do wzorca: lejący <= 0.98
|   |   |   |--- podobieństwo do wzorca: iglica <= 0.93
|   |   |   |   |--- podobieństwo do wzorca: pompa <= 0.92
|   |   |   |   |   |--- residual 7 kHz vs. baseline silnika <= -10.35
|   |   |   |   |   |   |--- class: unknown
|   |   |   |   |   |--- residual 7 kHz vs. baseline silnika >  -10.35
|   |   |   |   |   |   |--- class: pompa
|   |   |   |   |--- podobieństwo do wzorca: pompa >  0.92
|   |   |   |   |   |--- class: pompa
|   |   |   |--- podobieństwo do wzorca: iglica >  0.93
|   |   |   |   |--- residual 3 kHz vs. baseline silnika <= -4.77
|   |   |   |   |   |--- class: iglica
|   |   |   |   |--- residual 3 kHz vs. baseline silnika >  -4.77
|   |   |   |   |   |--- class: iglica
|   |   |--- podobieństwo do wzorca: lejący >  0.98
|   |   |   |--- podobieństwo do wzorca: zakoksowany <= -0.31
|   |   |   |   |--- clas

## Submit `test.csv` + zgodność nauczyciel / student

Ten sam nauczyciel (fit na `val.csv`) i student z `val` ∪ `train`. `test.csv` nie ma etykiet.

In [ ]:
sub_tree = model.predict(test)
sub_tree_train = model.predict(train)
sub_tabpfn = model.predict_teacher(test, model.teacher_)
sub_tabpfn_train = model.predict_teacher(train, model.teacher_)
sub_tree.to_csv("predictions_tree.csv", index=False)
sub_tree_train.to_csv("predictions_tree_train.csv", index=False)
sub_tabpfn.to_csv("predictions_tabpfn.csv", index=False)
sub_tabpfn_train.to_csv("predictions_tabpfn_train.csv", index=False)
agree = (sub_tree["label"] == sub_tabpfn["label"]).mean()
print(f"zgoda drzewo vs TabPFN na teście (bez etykiet): {agree:.3f}")
print("TabPFN\n", sub_tabpfn["label"].value_counts())
print("drzewo\n", sub_tree["label"].value_counts())
print("TabPFN na trainie\n", sub_tabpfn_train["label"].value_counts())
print("drzewo na trainie\n", sub_tree_train["label"].value_counts())

if IN_COLAB:
    files.download("predictions_tabpfn.csv")
    files.download("predictions_tree.csv")
    files.download("artifacts/diagnoser_tree.joblib")

zgoda drzewo vs TabPFN na teście (bez etykiet): 0.983
TabPFN
 label
ok             518
unknown         20
zakoksowany     17
pompa           16
iglica          16
lejacy          13
Name: count, dtype: int64
drzewo
 label
ok             515
pompa           22
unknown         21
zakoksowany     15
iglica          14
lejacy          13
Name: count, dtype: int64


Lokalnie po pobraniu `diagnoser_tree.joblib` do `artifacts/`:

```bash
streamlit run app_tabpfn.py
```

## (opcjonalnie) LOEO na `val ∪ final_valid`

Plików CSV **nie ruszamy**. W RAM sklejamy etykietowane zbiory, wstrzykujemy luki jak w `test.csv` i robimy leave-one-engine-out dla czterech modeli:

1. zwykłe drzewo ← labeled (bez TabPFN)
2. TabPFN teacher
3. drzewo destylowane z foldu + pseudo-`train.csv`
4. drzewo destylowane tylko z etykiet foldu

Na foldzie: szablony, nauczyciel i progi severity nie widzą held-out silnika. Na CPU to jest wolne (refit TabPFN × liczba silników). Ustaw `RUN_LOEO = True` i odpal komórkę. Przy `FINAL_EVALUATION_MODE=True` LOEO jest pomijane.


In [13]:
from tabpfn_diagnose import leave_one_engine_out, pick_device, read_labeled_csv
import pandas as pd

RUN_LOEO = False  # True → concat val ∪ final_valid w RAM, LOEO po silniku

if FINAL_EVALUATION_MODE:
    print("FINAL_EVALUATION_MODE: LOEO pominięte (final_valid jest w treningu)")
elif not RUN_LOEO:
    print("pominięte — ustaw RUN_LOEO = True")
else:
    val_loeo = read_labeled_csv("val.csv")
    ho_loeo = read_labeled_csv("final_valid.csv")
    train_loeo = pd.read_csv("train.csv")
    device_loeo = pick_device()
    n_est = 8 if device_loeo == "cuda" else 4
    labeled = pd.concat([val_loeo, ho_loeo], ignore_index=True)
    print("LOEO na val ∪ final_valid  (concat tylko w RAM, plików nie zapisujemy)")
    print(f"  silniki: {labeled['engine_id'].nunique()}  cylindry: {len(labeled)}")
    print(f"  z val: {len(val_loeo)}  z final_valid: {len(ho_loeo)}  device={device_loeo}")
    leave_one_engine_out(
        labeled,
        train_loeo,
        device=device_loeo,
        n_estimators=n_est,
        conf_min=0.70,
    )


FINAL_EVALUATION_MODE: LOEO pominięte (final_valid jest w treningu)
